In [1]:
!pip install transformers torch

In [2]:
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch

In [3]:
model_name = "microsoft/DialoGPT-medium"

print("Loading model...")

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name)

print("Model loaded successfully!")

Loading model...


Loading weights:   0%|          | 0/293 [00:00<?, ?it/s]

Model loaded successfully!


In [4]:
chat_history_ids = None

def generate_response(user_input):
    global chat_history_ids
    
    # Add assistant context
    context_input = "User: " + user_input + " Assistant:"
    
    # Encode user input
    new_input_ids = tokenizer.encode(context_input + tokenizer.eos_token, return_tensors='pt')
    
    # Combine conversation history
    if chat_history_ids is not None:
        input_ids = torch.cat([chat_history_ids, new_input_ids], dim=-1)
    else:
        input_ids = new_input_ids

    attention_mask = torch.ones(input_ids.shape, dtype=torch.long)

    # Generate response
    chat_history_ids = model.generate(
        input_ids,
        attention_mask=attention_mask,
        max_length=500,
        do_sample=True,
        top_k=40,
        top_p=0.9,
        temperature=0.7,
        repetition_penalty=1.2,
        no_repeat_ngram_size=2,
        pad_token_id=tokenizer.eos_token_id
    )

    # Decode response
    response = tokenizer.decode(
        chat_history_ids[:, input_ids.shape[-1]:][0],
        skip_special_tokens=True
    )

    return response

In [ ]:
print("Chatbot: Hello! I am your AI assistant. Type 'exit' to end the chat.")

while True:
    
    user_input = input("You: ")
    
    if user_input.lower() in ["exit", "quit"]:
        print("Chatbot: Goodbye! Have a nice day.")
        break
    
    response = generate_response(user_input)
    
    print("Chatbot:", response)

Chatbot: Hello! I am your AI assistant. Type 'exit' to end the chat.


You:  Hello


Chatbot: hello darkness my old friend...


You:  What is Artificial Intelligence?


Chatbot: Hello Darkness, my new friend.
